In [1]:
import scanpy as sc
import numpy as np
import scipy.sparse as sp
import logging
import anndata2ri
import rpy2.robjects as ro
from rpy2.robjects import r
from rpy2.robjects.conversion import localconverter
import rpy2.rinterface_lib.callbacks as rcb
import os

rcb.logger.setLevel(logging.ERROR)

In [2]:
path_template = ("../results/03_star/{sample}.Solo.out/{output}/{kind}/")

days = [0, 2, 5, 7]

samples = [
    "D2_C",
    "D2_D",
    "D5_A",
    "D5_B",
    "D5_C",
    "D7_C",
    "D7_D",
    "D0_A",
    "D0_B"
]


In [3]:
os.makedirs("../results/adata", exist_ok=True)

## Function to run SoupX

In [4]:
def run_soupx(filtered, raw):
    # Get clusters
    tmp = filtered.copy()
    sc.pp.normalize_total(tmp)
    sc.pp.log1p(tmp)
    sc.pp.highly_variable_genes(tmp)
    sc.pp.pca(tmp)
    sc.pp.neighbors(tmp)
    sc.tl.leiden(tmp, flavor="igraph")
    clusters = tmp.obs["leiden"]
    
    # Get cell and gene ids
    cells = filtered.obs_names
    genes = filtered.var_names
    droplets = raw.obs_names

    # Create matrix
    mat_filtered = sp.csc_matrix(filtered.X.T)
    mat_raw = sp.csc_matrix(raw.X.T)
    
    with localconverter(anndata2ri.converter):
        # Pass objects into R
        ro.globalenv["mat_filtered"] = mat_filtered
        ro.globalenv["mat_raw"] = mat_raw
        ro.globalenv["genes"] = genes
        ro.globalenv["cells"] = cells
        ro.globalenv["droplets"] = droplets
        ro.globalenv["clusters"] = clusters

        r("""
        suppressMessages(library(SoupX))

        rownames(mat_filtered) <- genes
        colnames(mat_filtered) <- cells

        rownames(mat_raw) <- genes
        colnames(mat_raw) <- droplets

        sc <- SoupChannel(mat_raw, mat_filtered)
        sc <- setClusters(sc, clusters)
        sc <- autoEstCont(sc, doPlot=FALSE)
        out <- adjustCounts(sc)
        rho <- sc$metaData$rho
        """)

        out = ro.globalenv["out"]
        rho = ro.globalenv["rho"]        
        filtered.layers["soupx"] = out.T
        return rho

## Process Data
Loop over each sample, do ambient RNA correction with Soupx, and concatenate data

In [5]:
from scipy.io import mmread

adata_list = []
for sample in samples:
    print(f"Processing sample: {sample}")  
    
    adata_filtered = sc.read_10x_mtx(
        path=path_template.format(sample=sample, output="Gene", kind="filtered"), 
        cache=True)
    
    adata_raw = sc.read_10x_mtx(
        path=path_template.format(sample=sample, output="Gene", kind="raw"), 
        cache=True)

    adata_filtered
    adata_raw
    
    velo_dir = path_template.format(sample=sample, output="Velocyto", kind="filtered")
    
    spliced = mmread(velo_dir + "spliced.mtx").T.tocsr()
    unspliced = mmread(velo_dir + "unspliced.mtx").T.tocsr()
    ambiguous = mmread(velo_dir + "ambiguous.mtx").T.tocsr()

    spliced.shape
    unspliced.shape
    ambiguous.shape

    adata_filtered.layers["spliced"] = spliced
    adata_filtered.layers["unspliced"] = unspliced
    adata_filtered.layers["ambiguous"] = ambiguous

    # Create some meta data columns
    adata_filtered.obs["sample"] = sample
    adata_filtered.obs["day"] = int(sample.lstrip("D").split("_")[0])

    # Run soupx to compute corrected count matrix
    rho = run_soupx(adata_filtered, adata_raw)
    print(f"Estimated contamination fraction of {rho[0]} for sample {sample}")

    adata_list.append(adata_filtered)


Processing sample: D2_C

    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    Estimated contamination fraction of 0.023 for sample D2_C
Processing sample: D2_D
Estimated contamination fraction of 0.022 for sample D2_D
Processing sample: D5_A
Estimated contamination fraction of 0.04 for sample D5_A
Processing sample: D5_B
Estimated contamination fraction of 0.025 for sample D5_B
Processing sample: D5_C
Estimated contamination fraction of 0.027 for sample D5_C
Processing sample: D7_C
Estimated contamination fraction of 0.019 for sample D7_C
Processing sample: D7_D
Estimated contamination fraction of 0.018000000000000002 for sample D7_D
Processing sample: D0_A
Estimated contamination fraction of 0.024 for sample D0_A
Processing sample: D0_B
Estimated contamination fraction of 0.046 for sample D0_B


In [6]:
# Concatenate
print("Concatenating data")

adata = sc.concat(adata_list)
adata.obs["sample"] = adata.obs["sample"].astype("category")
adata.obs["day"] = adata.obs["day"].astype("category").cat.reorder_categories(days)
 
# Create rounded soupx layer
adata.layers["soupx_rounded"] = np.round(adata.layers["soupx"])#.astype(np.int64) # Causes error with highly_variable_genes

# Store backup raw counts layer
adata.layers["raw_counts"] = adata.X

adata.obs_names_make_unique()

Concatenating data


/home/FCAM/kcobb/.miniforge/envs/single-cell/lib/python3.11/site-packages/anndata/_core/anndata.py:1811: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
adata.write("../results/adata/05-filter-ambient.h5ad")